# Evaluation of a structure of samples from models on CORDEX-ML_BENCH datasets

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.cordex_ml_default_params import *

In [ ]:
import functools
import math
import string

import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis.data import prep_eval_data
from mlde_analysis.psd import plot_psd, pysteps_rapsd
from mlde_analysis.display import pretty_table

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

## Figure: structure

* PSD

In [ ]:
# if len(eval_vars) > 1:
#     gridspec = np.pad(np.array(eval_vars), (0, -len(eval_vars) % 2), constant_values=".").reshape(-1, 1)
# else:
#     gridspec = np.array([eval_vars])
# structure_fig = plt.figure(figsize=(5.5*gridspec.shape[1], 3.5*gridspec.shape[0]), layout="constrained")
# axd = structure_fig.subplot_mosaic(gridspec, sharey=True, sharex=False)

for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    # gridspec = np.array([var]).reshape(1,1)
    structure_fig = plt.figure(figsize=(4, 3), layout="constrained")
    axd = structure_fig.subplot_mosaic([[var]], sharey=True, sharex=False)
    target_hr_rapsd = pysteps_rapsd(
        TARGET_DAS[var].
            stack(example=["ensemble_member", "time"]).cf.
            transpose("example", "Y", "X"),
        pixel_size=8.8
    ).mean(dim="example").drop_sel(freq=0)
    
    pred_rapsds = [
        {
            "label": model,
            "color": spec["color"],
            "data": pysteps_rapsd(
                EVAL_DS[source][f"pred_{var}"].sel(model=model).
                    stack(example=["ensemble_member", "sample_id", "time"]).cf.
                    transpose("example", "Y", "X"), 
                pixel_size=8.8).mean(dim="example").drop_sel(freq=0)
        }
        for source, mconfigs in MODELS.items() for model, spec in mconfigs.items()
    ]
    
    ax = axd[var]

    plot_psd(target_hr_rapsd, pred_rapsds, ax=ax)
    # ax.set_title(CPM_DAS[var].attrs["long_name"])
    
    plt.show()

    rapsd_errors_da = xr.concat([ 100*(h["data"] - target_hr_rapsd)/target_hr_rapsd for h in pred_rapsds ], dim="model").rename("raspd_error").assign_coords(wavelength=(["freq"], 1/target_hr_rapsd["freq"].values))

    fig = plt.figure(figsize=(4, 3), layout="constrained")
    ax = fig.subplots(1)
    l = rapsd_errors_da.plot(x="wavelength", hue="model", ax=ax)
    ax.set_title("RAPSD % error", fontsize="small")
    ax.set_xlim(10, 1000)
    ax.set_xscale("log")
    plt.rc('legend', fontsize = "xx-small", title_fontsize="x-small")
    
    plt.show()
    
    _ = pretty_table(rapsd_errors_da, round=4, dim_order=["freq", "model"])